# Testing my IOL-AI 2026 submission on Colab

This runs my submission — [`script.py`](https://huggingface.co/ritaberrada/iol-qwen25-14b-awq/blob/main/script.py) from `ritaberrada/iol-qwen25-14b-awq` — on Colab so I can **see what it does and check its answers before uploading to Hugging Face**.

The **prompt, model, and parsing are exactly what I submit**. Only three things change for Colab:

1. **Install the loader** — Colab has internet, so we `pip install` the AWQ loader; the offline sandbox already has it.
2. **Data** — the sandbox mounts a hidden test set at `/tmp/data/test.csv`; here we use **Linguini**, same format but *with reference answers*.
3. **Output** — instead of writing `submission.csv`, we compare the answers directly and score them.

Set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

## Setup

Installs the loader and pins `numpy`. Colab loads an older `numpy` at startup, so the runtime **restarts once** here for the new versions to take effect — when it comes back, just **Run all** again. (If it doesn't restart on its own: Runtime → Restart session, then Run all.)

In [ ]:
# Colab has internet, so we install the AWQ loader (gptqmodel) + deps. The offline
# sandbox installs nothing — autoawq is already there. numpy is pinned so the
# upgrade stays consistent.
!pip install -q -U transformers accelerate datasets sacrebleu gptqmodel "numpy==2.2.6"

# Colab loaded an older numpy at startup; the install just replaced it on disk, so we
# restart the runtime once to load the new one. It stops here — then Run all again and
# this check passes straight through.
import numpy, IPython
if numpy.__version__ != "2.2.6":
    print("Installed — restarting the runtime. When it's back, hit Run all again.", flush=True)
    IPython.Application.instance().kernel.do_shutdown(True)

## 1. Load the model

Same as the submission, with one change: on Colab the weights come from the Hub, so `MODEL_ID` is the repo name. In the sandbox `MODEL_ID = "."` (the weights ship inside the repo) and it forces offline mode — we don't, because here we download from the Hub.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "ritaberrada/iol-qwen25-14b-awq"   # in the sandbox this is "." — the weights ship in the repo
MAX_NEW_TOKENS = 1536                          # room to reason; lower = faster but answers may get cut off

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()
print("loaded", MODEL_ID, "| VRAM", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")

## 2. The prompt and the parser

**Identical to the submission** — this is exactly how the model is asked and how its answer is read back. Nothing here changes between Colab and the sandbox.

In [ ]:
import re

SYSTEM = (
    "You solve International Linguistics Olympiad problems by reasoning from the "
    "data given. Reason step by step. Then write a line that says exactly "
    "FINAL ANSWERS: and, below it, one answer per line in the order the items are "
    "asked -- the bare answer only, no numbering, no quotes, no extra text."
)

def parse_answers(text):
    """Keep only the lines after the last 'FINAL ANSWERS:' marker, one answer per line."""
    marker = list(re.finditer(r"(?im)^\s*final answers?\s*:?\s*$", text))
    if marker:
        text = text[marker[-1].end():]
    answers = []
    for line in text.splitlines():
        line = re.sub(r"^\s*\d+[.)]\s*", "", line).strip()   # drop "1. " / "2) " if the model adds it
        if line:
            answers.append(line)
    return answers

## 3. The test data

The one real swap. The sandbox mounts a **hidden** test set at `/tmp/data/test.csv`. On Colab we can't see that, so we use **Linguini** — same `context` / `query` columns, but it also ships the reference answers, which is what lets us check the script below.

In [ ]:
from datasets import load_dataset

# Sandbox: df = pd.read_csv("/tmp/data/test.csv", dtype=str).fillna("")   # the hidden test set
# Colab:   Linguini — same columns (context, query) PLUS reference answers to check against.
ds = load_dataset("facebook/linguini")
df = ds["test"].to_pandas().sample(8, random_state=0).reset_index(drop=True)
print(len(df), "problems |", list(df["task_type"].unique()))

## 4. Run the script

The submission's generation loop, unchanged: build the chat message, generate greedily, parse the answers. The only difference is the last line — instead of writing `submission.csv`, we keep each raw output and its answers in memory so we can look at them and score them.

In [ ]:
results = []
for i, r in df.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{r['context'].strip()}\n\n{r['query'].strip()}"},
    ]
    ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    text = tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    answers = parse_answers(text)
    # Sandbox appends {"id", "pred": json.dumps(answers)} to submission.csv;
    # here we keep the raw output + answers + reference so we can look and score.
    results.append({"query": r["query"], "reference": r["answer"], "raw": text, "pred": answers})
    print(f"[{i + 1}/{len(df)}] {len(answers)} answers", flush=True)

## 5. See what the script does

For each problem: the raw text the model generated, the answers the parser pulled out, and the reference. This is the whole point of testing on Colab — watching what your script actually produces.

In [ ]:
for i, res in enumerate(results, 1):
    print("=" * 72)
    print(f"PROBLEM {i}")
    print("=" * 72)
    print(res["query"][:300])
    print("\n--- RAW MODEL OUTPUT ---")
    print(res["raw"])
    print("\nPARSED ANSWERS:", res["pred"])
    print("REFERENCE:     ", res["reference"])
    print()

## 6. Check the answers

Because Linguini ships the reference answers, we can score the predictions the same way the competition does — exact match and chrF. The sandbox can't do this; it only writes `submission.csv` for the platform to score.

In [ ]:
import ast
from sacrebleu.metrics import CHRF
chrf = CHRF()

def gold_alts(reference):
    """Reference: a list of items, each a list of accepted alternatives."""
    v = ast.literal_eval(str(reference))
    return [[str(a) for a in it] if isinstance(it, (list, tuple)) else [str(it)] for it in v]

em = cf = n = 0
for res in results:
    gold = gold_alts(res["reference"])
    preds = (res["pred"] + [""] * len(gold))[:len(gold)]   # line up by position, as the scorer does
    for p, alts in zip(preds, gold):
        em += any(p.strip().lower() == a.strip().lower() for a in alts)
        cf += max(chrf.sentence_score(p, [a]).score for a in alts)
        n += 1

print(f"items scored: {n}")
print(f"exact match:  {em / n:.2f}")
print(f"chrF:         {cf / n:.1f}")